In [121]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm #need to install statsmodels
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import dataframe_image as dfi

In [122]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

In [123]:
df_assess = pd.read_csv("data/cleaned/final_assessment.csv")
df_grades = pd.read_csv("data/cleaned/grades.csv")
df_status = pd.read_csv("data/cleaned/status.csv")
df_intake = pd.read_csv("data/cleaned/intake_form.csv")

In [124]:
df_intake

,id,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,DSC 140B,major,neither,Second year,"MATH 180A, MATH 181A",4,2,5
1,1,DSC 140B,major,neither,Third year,MATH 180A,4,3,5
2,3,DSC 140B,major,neither,Third year,MATH 183,4,3,4
3,6,DSC 140B,major,neither,Second year,MATH 183,3,1,4
4,9,DSC 140B,major,major,Third year,MATH 183,4,2,4
5,11,DSC 140B,major,neither,Second year,MATH 183,3,1,4
6,12,DSC 140B,major,neither,Third year,MATH 183,4,4,4
7,15,DSC 80,major,neither,Third year (first year transfer),MATH 183,3,1,4
8,20,DSC 180,major,neither,Third year (first year transfer),MATH 183,2,2,4
9,24,DSC 180,major,neither,Fourth year (second year transfer),"MATH 180A, MATH 181A, MATH 183",5,5,5


## Merge two dataframes(grades and status) by id, create coding and handwritten binary columns for b1 vs b2 test. I got rid of section 4 for now since they will affect both b1(coefficient of coding) and b2(coefficient of handwritten).

In [125]:
df_status = df_status[df_status['completed'] == 1]

sample = df_grades.merge(df_status[["id", "section"]], on="id", how="inner")

sample['coding'] =  ((sample['section'] == 2) | (sample['section'] == 4)).astype(int) #(yes,1), (no,0)
sample['handwritten'] =  ((sample['section'] == 3) | (sample['section'] == 4)).astype(int) #(yes,1), (no,0)
#sample_filter = sample[(sample['coding'] == 0) | (sample['handwritten'] == 0)]
sample

,id,coding_score,handwritten_score,final_score,final_score_adj,section,coding,handwritten
0,0,NaN,NaN,0.737705,0.616393,1,0,0
1,1,0.928571,NaN,0.754098,0.639344,2,1,0
2,3,0.857143,1.0,0.836066,0.590164,4,1,1
3,6,NaN,1.0,0.508197,0.262295,3,0,1
4,9,0.904764,NaN,0.672131,0.511475,2,1,0
5,11,0.928571,1.0,0.918033,0.439344,4,1,1
6,12,NaN,NaN,0.098361,0.019672,1,0,0
7,15,1.000000,1.0,0.344262,0.196721,4,1,1
8,20,NaN,NaN,0.573770,0.357377,1,0,0
9,24,NaN,NaN,1.000000,0.803279,1,0,0


## Linear model for final_score = bias(section1) + b1 * coding + b2 * handwritten + b3 * (coding*written)

In [126]:
res = smf.ols(
    formula='final_score ~ coding * handwritten',
    data=sample
).fit()

print(res.summary())

s1 = res.summary2()
df_model = s1.tables[0]
df_coef  = s1.tables[1]
dfi.export(df_coef,  "data/statistic_analysis_outputs/final_separate.png", table_conversion="chrome", dpi = 300)

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                 -0.056
Method:                 Least Squares   F-statistic:                    0.4666
Date:                Sun, 01 Mar 2026   Prob (F-statistic):              0.708
Time:                        15:27:30   Log-Likelihood:                 3.5034
No. Observations:                  31   AIC:                            0.9932
Df Residuals:                      27   BIC:                             6.729
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.5961      0

## Linear model is final score = bias(section 1) + b1 * (handwritten + coding) + b2 * (coding * written)

In [127]:
res2 = smf.ols(
    formula='final_score ~ I(coding + handwritten) + coding:handwritten',
    data=sample
).fit()

print(res2.summary())
#anova_results = anova_lm(model2, model1)
s2 = res2.summary2()
df_model2 = s2.tables[0]
df_coef2  = s2.tables[1]
dfi.export(df_coef2,  "data/statistic_analysis_outputs/final_combine.png", table_conversion="chrome", dpi = 300)

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                 -0.019
Method:                 Least Squares   F-statistic:                    0.7162
Date:                Sun, 01 Mar 2026   Prob (F-statistic):              0.497
Time:                        15:27:32   Log-Likelihood:                 3.4933
No. Observations:                  31   AIC:                           -0.9867
Df Residuals:                      28   BIC:                             3.315
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

## Anova Test between two models

In [128]:
anova_results = sm.stats.anova_lm(res2, res)
print(anova_results)
dfi.export(
    anova_results,
    "data/statistic_analysis_outputs/final_anova.png",
    table_conversion="chrome",
    dpi = 300
)

   df_resid       ssr  df_diff   ss_diff         F    Pr(>F)
0      28.0  1.448797      0.0       NaN       NaN       NaN
1      27.0  1.447856      1.0  0.000941  0.017541  0.895617


## Linear model for adjusted final score = bias(section1) + b1 * coding + b2 * handwritten + b3 * (coding*written)

In [129]:
res3 = smf.ols(
    formula='final_score_adj ~ coding * handwritten',
    data=sample
).fit()

print(res3.summary())

s3 = res3.summary2()
df_model3 = s3.tables[0]
df_coef3  = s3.tables[1]
dfi.export(df_coef3,  "data/statistic_analysis_outputs/final_adj_separate.png", table_conversion="chrome", dpi = 300)


                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.006
Model:                            OLS   Adj. R-squared:                 -0.104
Method:                 Least Squares   F-statistic:                   0.05768
Date:                Sun, 01 Mar 2026   Prob (F-statistic):              0.981
Time:                        15:27:34   Log-Likelihood:                 5.5015
No. Observations:                  31   AIC:                            -3.003
Df Residuals:                      27   BIC:                             2.733
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.4340      0

## Linear model is adjusted final score = bias(section 1) + b1 * (handwritten + coding) + b2 * (coding * written)

In [130]:
res4 = smf.ols(
    formula='final_score_adj ~ I(coding + handwritten) + coding:handwritten',
    data=sample
).fit()

print(res4.summary())
#anova_results = anova_lm(model2, model1)
s4 = res4.summary2()
df_model4 = s4.tables[0]
df_coef4  = s4.tables[1]
dfi.export(df_coef4,  "data/statistic_analysis_outputs/final_adj_combine.png", table_conversion="chrome", dpi = 300)

                            OLS Regression Results                            
Dep. Variable:        final_score_adj   R-squared:                       0.006
Model:                            OLS   Adj. R-squared:                 -0.065
Method:                 Least Squares   F-statistic:                   0.08806
Date:                Sun, 01 Mar 2026   Prob (F-statistic):              0.916
Time:                        15:27:36   Log-Likelihood:                 5.4996
No. Observations:                  31   AIC:                            -4.999
Df Residuals:                      28   BIC:                           -0.6973
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
Intercept                 

## Anova Test between two models for adjusted final score

In [131]:
anova_results2 = sm.stats.anova_lm(res4, res3)
print(anova_results2)
dfi.export(
    anova_results2,
    "data/statistic_analysis_outputs/final_adj_anova.png",
    table_conversion="chrome",
    dpi = 300
)

   df_resid       ssr  df_diff  ss_diff         F    Pr(>F)
0      28.0  1.272895      0.0      NaN       NaN       NaN
1      27.0  1.272745      1.0  0.00015  0.003193  0.955357


## Including variables in intake form

In [132]:
new_sample = sample.merge(df_intake, on="id", how="inner")
sample_new = new_sample.drop(columns = ['coding_score', 'handwritten_score', 'section'])
sample_new

,id,final_score,final_score_adj,coding,handwritten,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,0.737705,0.616393,0,0,DSC 140B,major,neither,Second year,"MATH 180A, MATH 181A",4,2,5
1,1,0.754098,0.639344,1,0,DSC 140B,major,neither,Third year,MATH 180A,4,3,5
2,3,0.836066,0.590164,1,1,DSC 140B,major,neither,Third year,MATH 183,4,3,4
3,6,0.508197,0.262295,0,1,DSC 140B,major,neither,Second year,MATH 183,3,1,4
4,9,0.672131,0.511475,1,0,DSC 140B,major,major,Third year,MATH 183,4,2,4
5,11,0.918033,0.439344,1,1,DSC 140B,major,neither,Second year,MATH 183,3,1,4
6,12,0.098361,0.019672,0,0,DSC 140B,major,neither,Third year,MATH 183,4,4,4
7,15,0.344262,0.196721,1,1,DSC 80,major,neither,Third year (first year transfer),MATH 183,3,1,4
8,20,0.573770,0.357377,0,0,DSC 180,major,neither,Third year (first year transfer),MATH 183,2,2,4
9,24,1.000000,0.803279,0,0,DSC 180,major,neither,Fourth year (second year transfer),"MATH 180A, MATH 181A, MATH 183",5,5,5


In [133]:
sample_new['recruitment_source'].value_counts()
#If I remember it right, class_standing(student's year) is redundant with this feature
#since older students are likely to recruited from a uppertive class

heard_type = {
    'DSC 10': 'Heard_1',
    'DSC 20': 'Heard_1',
    'DSC 30': 'Heard_1',
    'DSC 40A': 'Heard_2',
    'DSC 40B': 'Heard_2',
    'DSC 80': 'Heard_2',
    'DSC 140B': 'Heard_3',
    'CSE 150A': 'Heard_3',
    'CSE 151A': 'Heard_3',
    'DSC 180': 'Heard_3',
    'DSC 100': 'Heard_4',
    'DSC 120': 'Heard_4',
    'DSC 190': 'Heard_4',
}

In [134]:
sample_new['dsc_affiliation'].value_counts()
dsc_type = {
    'major': 'DSC_major',
    'minor': 'DSC_minor',
    'neither': 'DSC_neither',
}

In [135]:
sample_new['math_affiliation'].value_counts()

math_type = {
    'major': 'MATH_major',
    'minor': 'MATH_minor',
    'neither': 'MATH_neither',
}

In [136]:
sample_new['class_standing'].value_counts()
year_type = {
    'First year': 'First_year',
    'Second year': 'Second_year',
    'Third year': 'Third_year',
    'Fourth year': 'Fourth_year',
    'Third year (first year transfer)': 'First_year_transfer',
    'Fourth year (second year transfer)': 'Second_year_transfer'
}

In [137]:
sample_new['stats_courses_taken'].value_counts()

section_type = {
    'MATH 181A': 'Stats_1',
    'MATH 180A': 'Stats_2',
    'ECE 109': 'Stats_2',
    'MAE 108': 'Stats_2',
    'MATH 183': 'Stats_3',
    'MATH 186': 'Stats_3',
    'ECON 120A': 'Stats_3',
    'None of the above': 'Stats_4'
}

In [138]:
def mappings(col, dictionary):
    values = [i.strip() for i in col.split(',')]
    adjust = {dictionary.get(j) for j in values if j in dictionary} #keep unique values
    return ', '.join(adjust)

In [139]:
samples = sample_new.copy()
samples['recruitment_source'] = samples['recruitment_source'].apply(lambda x: mappings(x, heard_type))
samples['stats_courses_taken'] = samples['stats_courses_taken'].apply(lambda x: mappings(x, section_type))
samples['dsc_affiliation'] = samples['dsc_affiliation'].apply(lambda x: mappings(x, dsc_type))
samples['math_affiliation'] = samples['math_affiliation'].apply(lambda x: mappings(x, math_type))
samples['class_standing'] = samples['class_standing'].apply(lambda x: mappings(x, year_type))
samples

,id,final_score,final_score_adj,coding,handwritten,recruitment_source,dsc_affiliation,math_affiliation,class_standing,stats_courses_taken,stats_confidence,chebyshev_familiarity,python_skill_level
0,0,0.737705,0.616393,0,0,Heard_3,DSC_major,MATH_neither,Second_year,"Stats_2, Stats_1",4,2,5
1,1,0.754098,0.639344,1,0,Heard_3,DSC_major,MATH_neither,Third_year,Stats_2,4,3,5
2,3,0.836066,0.590164,1,1,Heard_3,DSC_major,MATH_neither,Third_year,Stats_3,4,3,4
3,6,0.508197,0.262295,0,1,Heard_3,DSC_major,MATH_neither,Second_year,Stats_3,3,1,4
4,9,0.672131,0.511475,1,0,Heard_3,DSC_major,MATH_major,Third_year,Stats_3,4,2,4
5,11,0.918033,0.439344,1,1,Heard_3,DSC_major,MATH_neither,Second_year,Stats_3,3,1,4
6,12,0.098361,0.019672,0,0,Heard_3,DSC_major,MATH_neither,Third_year,Stats_3,4,4,4
7,15,0.344262,0.196721,1,1,Heard_2,DSC_major,MATH_neither,First_year_transfer,Stats_3,3,1,4
8,20,0.573770,0.357377,0,0,Heard_3,DSC_major,MATH_neither,First_year_transfer,Stats_3,2,2,4
9,24,1.000000,0.803279,0,0,Heard_3,DSC_major,MATH_neither,Second_year_transfer,"Stats_2, Stats_1, Stats_3",5,5,5


In [140]:
df_encoded = pd.concat([samples, samples['stats_courses_taken'].str.get_dummies(sep=', '), 
                        samples['recruitment_source'].str.get_dummies(sep=', '),
                        samples['dsc_affiliation'].str.get_dummies(sep=', '),
                        samples['math_affiliation'].str.get_dummies(sep=', '),
                        samples['class_standing'].str.get_dummies(sep=', '),], axis = 1)
df_encoded = df_encoded.drop(columns = ['recruitment_source', 'dsc_affiliation', 'math_affiliation', 'stats_courses_taken', 'Stats_4',
                           'DSC_neither', 'MATH_neither', 'class_standing'], errors='ignore') 
#dropping all string columns and neither columns to prevent redundency

#needed for backward elimination, this is for people who are in section 4, who coding and handwritten problem sets
df_encoded['coding_handwritten'] = df_encoded['coding']*df_encoded['handwritten']
#reorder the columns
df_encoded = df_encoded[list(df_encoded.columns[:5]) + ['coding_handwritten'] + list(df_encoded.columns[5:-1])]
df_encoded

,id,final_score,final_score_adj,coding,handwritten,coding_handwritten,stats_confidence,chebyshev_familiarity,python_skill_level,Stats_1,Stats_2,Stats_3,Heard_1,Heard_2,Heard_3,Heard_4,DSC_major,DSC_minor,MATH_major,First_year,First_year_transfer,Fourth_year,Second_year,Second_year_transfer,Third_year
0,0,0.737705,0.616393,0,0,0,4,2,5,1,1,0,0,0,1,0,1,0,0,0,0,0,1,0,0
1,1,0.754098,0.639344,1,0,0,4,3,5,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,1
2,3,0.836066,0.590164,1,1,1,4,3,4,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,1
3,6,0.508197,0.262295,0,1,0,3,1,4,0,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0
4,9,0.672131,0.511475,1,0,0,4,2,4,0,0,1,0,0,1,0,1,0,1,0,0,0,0,0,1
5,11,0.918033,0.439344,1,1,1,3,1,4,0,0,1,0,0,1,0,1,0,0,0,0,0,1,0,0
6,12,0.098361,0.019672,0,0,0,4,4,4,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,1
7,15,0.344262,0.196721,1,1,1,3,1,4,0,0,1,0,1,0,0,1,0,0,0,1,0,0,0,0
8,20,0.573770,0.357377,0,0,0,2,2,4,0,0,1,0,0,1,0,1,0,0,0,1,0,0,0,0
9,24,1.000000,0.803279,0,0,0,5,5,5,1,1,1,0,0,1,0,1,0,0,0,0,0,0,1,0


In [141]:
df_encoded.columns

Index(['id', 'final_score', 'final_score_adj', 'coding', 'handwritten', 'coding_handwritten', 'stats_confidence', 'chebyshev_familiarity',
       'python_skill_level', 'Stats_1', 'Stats_2', 'Stats_3', 'Heard_1', 'Heard_2', 'Heard_3', 'Heard_4', 'DSC_major', 'DSC_minor', 'MATH_major', 'First_year',
       'First_year_transfer', 'Fourth_year', 'Second_year', 'Second_year_transfer', 'Third_year'],
      dtype='object')

## Moving on with backward elimnination, we don't want our model to overfit.

## Here are all of the input features we will use to calculate the final_score and final_score_adj

In [142]:
initial_vars = [col for col in df_encoded.columns if col != "id" and col != 'final_score' and col != 'final_score_adj']
initial_vars

['coding',
 'handwritten',
 'coding_handwritten',
 'stats_confidence',
 'chebyshev_familiarity',
 'python_skill_level',
 'Stats_1',
 'Stats_2',
 'Stats_3',
 'Heard_1',
 'Heard_2',
 'Heard_3',
 'Heard_4',
 'DSC_major',
 'DSC_minor',
 'MATH_major',
 'First_year',
 'First_year_transfer',
 'Fourth_year',
 'Second_year',
 'Second_year_transfer',
 'Third_year']

## There are variables that we need to keep in our experiment, there are:

In [143]:
forced_vars = ['coding', 'handwritten', 'coding_handwritten']

In [144]:
def backward_selection(data, initial_vars, response, forced_vars, p_thresh=0.5):
    """
    Backward elimination with forced variables retained.
    
    Parameters
    ----------
    data : DataFrame
    initial_vars : list
        Input variables
    response : str
    forced_vars : list
        Variables that must remain in the model
    p_thresh : float
        Threshold for p-value removal
    
    Returns
    -------
    final_model : fitted statsmodels model
    """
    
    # create a copy 
    remaining = initial_vars.copy()
    count = 0
    
    while True:
        formula = response + " ~ " + " + ".join(remaining)
        model = smf.ols(formula=formula, data=data).fit()
        
        #We need to keep intercept and force variables, so we won't consider these variables in the elimination
        pvalues = model.pvalues.drop("Intercept")
        pvalues = pvalues.drop(labels=forced_vars, errors='ignore')
        
        max_p = pvalues.max()
        
        if max_p > p_thresh:
            count += 1
            remove_var = pvalues.idxmax()
            print(f"\n This is the {count} iteration of backward elimination")
            print(f"Removing {remove_var} (p={max_p:.4f})")
            remaining.remove(remove_var)
            print("\n ===================================================================================")
        else:
            break
    
    final_formula = response + " ~ " + " + ".join(remaining)
    final_model = smf.ols(formula=final_formula, data=data).fit()
    
    print(f"The backward elimination has ran {count} iterations")
    print("\nThis is the final formula:")
    print(final_formula)
    
    return final_model.summary()

## This is the backward elimination with final_score as the predict variable with final model at the end.

In [145]:
backward_selection(df_encoded, initial_vars, 'final_score', forced_vars, p_thresh=0.5)


 This is the 1 iteration of backward elimination
Removing python_skill_level (p=0.9951)


 This is the 2 iteration of backward elimination
Removing chebyshev_familiarity (p=0.9892)


 This is the 3 iteration of backward elimination
Removing First_year_transfer (p=0.8481)


 This is the 4 iteration of backward elimination
Removing Third_year (p=0.8585)


 This is the 5 iteration of backward elimination
Removing Stats_2 (p=0.7939)


 This is the 6 iteration of backward elimination
Removing Stats_1 (p=0.8056)


 This is the 7 iteration of backward elimination
Removing MATH_major (p=0.6960)


 This is the 8 iteration of backward elimination
Removing Fourth_year (p=0.7973)


 This is the 9 iteration of backward elimination
Removing DSC_minor (p=0.6456)


 This is the 10 iteration of backward elimination
Removing Heard_3 (p=0.7634)


 This is the 11 iteration of backward elimination
Removing DSC_major (p=0.5300)

The backward elimination has ran 11 iterations

This is the final formula:
fin

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            final_score   R-squared:                       0.510
Model:                            OLS   Adj. R-squared:                  0.226
Method:                 Least Squares   F-statistic:                     1.796
Date:                Sun, 01 Mar 2026   Prob (F-statistic):              0.126
Time:                        15:27:39   Log-Likelihood:                 13.771
No. Observations:                  31   AIC:                            -3.543
Df Residuals:                      19   BIC:                             13.67
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
========================================================================================
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.7030      0.241      2.912      0.009       0.198       1.208
coding                   0.1408      0.121      1.166      0.258      -0.112       0.393
handwritten             -0.0205      0.112     -0.183      0.856      -0.254       0.213
coding_handwritten       0.1234      0.180      0.685      0.502      -0.254       0.500
stats_confidence        -0.0506      0.051     -0.998      0.331      -0.157       0.055
Stats_3                 -0.0730      0.096     -0.759      0.457      -0.274       0.128
Heard_1                  0.5702      0.258      2.206      0.040       0.029       1.111
Heard_2                 -0.3040      0.172     -1.770      0.093      -0.663       0.055
Heard_4                  0.3912      0.148      2.649      0.016       0.082       0.700
First_year              -0.5969      0.265     -2.253      0.036      -1.151      -0.042
Second_year              0.1856      0.132      1.406      0.176      -0.091       0.462
Second_year_transfer     0.4683      0.173      2.712      0.014       0.107       0.830
==============================================================================
Omnibus:                        0.794   Durbin-Watson:                   2.205
Prob(Omnibus):                  0.672   Jarque-Bera (JB):                0.497
Skew:                          -0.307   Prob(JB):                        0.780
Kurtosis:                       2.912   Cond. No.                         41.1
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## This is the backward elimination with final_score_adj as the predict variable with final model at the end.

In [146]:
backward_selection(df_encoded, initial_vars, 'final_score_adj', forced_vars, p_thresh=0.5)


 This is the 1 iteration of backward elimination
Removing Stats_1 (p=0.9904)


 This is the 2 iteration of backward elimination
Removing Heard_3 (p=0.9311)


 This is the 3 iteration of backward elimination
Removing chebyshev_familiarity (p=0.8952)


 This is the 4 iteration of backward elimination
Removing Stats_3 (p=0.9111)


 This is the 5 iteration of backward elimination
Removing First_year_transfer (p=0.8457)


 This is the 6 iteration of backward elimination
Removing python_skill_level (p=0.8019)


 This is the 7 iteration of backward elimination
Removing MATH_major (p=0.7501)


 This is the 8 iteration of backward elimination
Removing DSC_minor (p=0.5996)


 This is the 9 iteration of backward elimination
Removing DSC_major (p=0.6287)

The backward elimination has ran 9 iterations

This is the final formula:
final_score_adj ~ coding + handwritten + coding_handwritten + stats_confidence + Stats_2 + Heard_1 + Heard_2 + Heard_4 + First_year + Fourth_year + Second_year + Second_ye

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:        final_score_adj   R-squared:                       0.563
Model:                            OLS   Adj. R-squared:                  0.229
Method:                 Least Squares   F-statistic:                     1.685
Date:                Sun, 01 Mar 2026   Prob (F-statistic):              0.155
Time:                        15:27:39   Log-Likelihood:                 18.233
No. Observations:                  31   AIC:                            -8.466
Df Residuals:                      17   BIC:                             11.61
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
========================================================================================
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.4307      0.208      2.075      0.053      -0.007       0.869
coding                   0.0572      0.123      0.463      0.649      -0.203       0.318
handwritten             -0.0501      0.112     -0.447      0.661      -0.287       0.187
coding_handwritten       0.0921      0.178      0.518      0.611      -0.283       0.468
stats_confidence        -0.0630      0.059     -1.063      0.303      -0.188       0.062
Stats_2                  0.1908      0.124      1.539      0.142      -0.071       0.452
Heard_1                  0.5713      0.250      2.281      0.036       0.043       1.100
Heard_2                 -0.1263      0.165     -0.767      0.453      -0.474       0.221
Heard_4                  0.2227      0.148      1.509      0.150      -0.089       0.534
First_year              -0.4937      0.270     -1.829      0.085      -1.063       0.076
Fourth_year              0.1860      0.181      1.029      0.318      -0.195       0.567
Second_year              0.1446      0.146      0.988      0.337      -0.164       0.454
Second_year_transfer     0.4283      0.213      2.010      0.061      -0.021       0.878
Third_year               0.1297      0.182      0.714      0.485      -0.254       0.513
==============================================================================
Omnibus:                        1.455   Durbin-Watson:                   2.074
Prob(Omnibus):                  0.483   Jarque-Bera (JB):                0.898
Skew:                          -0.417   Prob(JB):                        0.638
Kurtosis:                       3.021   Cond. No.                         44.2
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""